In [ ]:
# Return Analysis and Asset Risk Characteristics

This notebook analyzes historical return characteristics of the ETF universe.

The analysis includes:

- Daily return calculation
- Annualized return and volatility
- Drawdown analysis
- Distribution characteristics
- Correlation structure
- Annualized covariance estimation

These statistics provide the empirical foundation for portfolio optimization models.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# 1. Load adjusted prices
# =========================

data_dir = Path("../data")

prices = pd.read_csv(
    data_dir / "adjusted_close.csv",
    index_col=0,
    parse_dates=True
)

print("Price data shape:", prices.shape)
print("Date range:", prices.index.min(), "→", prices.index.max())


# =========================
# 2. Daily returns
# =========================

returns = prices.pct_change().dropna()

print("\nReturn data shape:", returns.shape)

returns.to_csv(data_dir / "daily_returns.csv")


# =========================
# 3. Descriptive statistics
# =========================

TRADING_DAYS = 252

annual_return = returns.mean() * TRADING_DAYS
annual_volatility = returns.std() * np.sqrt(TRADING_DAYS)

# Historical compound annual growth rate
years = (prices.index[-1] - prices.index[0]).days / 365.25

cagr = (prices.iloc[-1] / prices.iloc[0]) ** (1 / years) - 1

# Maximum drawdown of each individual asset
wealth = (1 + returns).cumprod()
running_max = wealth.cummax()
drawdown = wealth / running_max - 1
max_drawdown = drawdown.min()

summary = pd.DataFrame({
    "CAGR": cagr,
    "Annualized_Mean_Return": annual_return,
    "Annualized_Volatility": annual_volatility,
    "Max_Drawdown": max_drawdown,
    "Skewness": returns.skew(),
    "Excess_Kurtosis": returns.kurt()
})

summary = summary.sort_values(
    "Annualized_Volatility"
)

summary.to_csv(data_dir / "asset_summary_stats.csv")

print("\n=== Asset Summary Statistics ===")
display(summary.round(4))


# =========================
# 4. Correlation matrix
# =========================

correlation = returns.corr()

correlation.to_csv(
    data_dir / "return_correlation.csv"
)

print("\n=== Return Correlation Matrix ===")
display(correlation.round(3))


# =========================
# 5. Covariance matrix
# =========================

annual_covariance = returns.cov() * TRADING_DAYS

annual_covariance.to_csv(
    data_dir / "annual_covariance.csv"
)

print("\n=== Annualized Covariance Matrix ===")
display(annual_covariance.round(4))


# =========================
# 6. Data validation
# =========================

print("\n=== Validation ===")
print("Missing returns:", returns.isna().sum().sum())
print("Infinite returns:", np.isinf(returns).sum().sum())
print("Largest daily return:", returns.max().max())
print("Smallest daily return:", returns.min().min())

Price data shape: (4024, 8)
Date range: 2010-01-04 00:00:00 → 2025-12-31 00:00:00

Return data shape: (4023, 8)

=== Asset Summary Statistics ===


,CAGR,Annualized_Mean_Return,Annualized_Volatility,Max_Drawdown,Skewness,Excess_Kurtosis
TLT,0.0273,0.0384,0.1509,-0.4835,0.0469,3.5131
GLD,0.0836,0.0930,0.1582,-0.4556,-0.4207,4.3662
SPY,0.1391,0.1453,0.1721,-0.3370,-0.3331,12.1135
EFA,0.0650,0.0801,0.1841,-0.3419,-0.5872,9.3350
VNQ,0.0855,0.1033,0.2050,-0.4242,-0.7416,15.5825
QQQ,0.1856,0.1919,0.2064,-0.3512,-0.2087,7.3174
EEM,0.0375,0.0597,0.2130,-0.3980,-0.3542,6.0760
IWM,0.1029,0.1231,0.2230,-0.4113,-0.4089,6.6705



=== Return Correlation Matrix ===


,SPY,QQQ,IWM,EFA,EEM,TLT,GLD,VNQ
SPY,1.000,0.932,0.884,0.857,0.782,-0.298,0.053,0.748
QQQ,0.932,1.000,0.796,0.760,0.731,-0.231,0.053,0.617
IWM,0.884,0.796,1.000,0.796,0.727,-0.272,0.060,0.746
EFA,0.857,0.760,0.796,1.000,0.848,-0.289,0.142,0.689
EEM,0.782,0.731,0.727,0.848,1.000,-0.255,0.176,0.609
TLT,-0.298,-0.231,-0.272,-0.289,-0.255,1.000,0.217,-0.096
GLD,0.053,0.053,0.060,0.142,0.176,0.217,1.000,0.116
VNQ,0.748,0.617,0.746,0.689,0.609,-0.096,0.116,1.000



=== Annualized Covariance Matrix ===


,SPY,QQQ,IWM,EFA,EEM,TLT,GLD,VNQ
SPY,0.0296,0.0331,0.0339,0.0272,0.0287,-0.0077,0.0014,0.0264
QQQ,0.0331,0.0426,0.0366,0.0289,0.0322,-0.0072,0.0017,0.0261
IWM,0.0339,0.0366,0.0497,0.0327,0.0345,-0.0091,0.0021,0.0341
EFA,0.0272,0.0289,0.0327,0.0339,0.0333,-0.0080,0.0041,0.0260
EEM,0.0287,0.0322,0.0345,0.0333,0.0454,-0.0082,0.0059,0.0266
TLT,-0.0077,-0.0072,-0.0091,-0.0080,-0.0082,0.0228,0.0052,-0.0030
GLD,0.0014,0.0017,0.0021,0.0041,0.0059,0.0052,0.0250,0.0038
VNQ,0.0264,0.0261,0.0341,0.0260,0.0266,-0.0030,0.0038,0.0420



=== Validation ===
Missing returns: 0
Infinite returns: 0
Largest daily return: 0.12003076479350083
Smallest daily return: -0.17727730246539042
